In [1]:
import os
os.chdir('../')

In [4]:
import numpy as np
import torch
from pathlib import Path
from PIL import Image, ImageOps
from utils.clean_fid import CleanFIDInception
from tqdm import tqdm

# ------------ config ------------
DEVICE    = torch.device('cuda:0')
VAL_NPZ   = 'prompts/mscoco2014_valid_30k.npz'
IMG_ROOT  = Path('/dataset/val2014')
OUT_DIR   = Path('mscoco2014_fid')
OUT_PATH  = OUT_DIR / 'coco2014_val_30k_fid_stats_clean.pt'

BATCH_SIZE    = 64      # Inception 추출 배치 크기 (FIDInception가 리스트 입력을 받는 가정)
LIMIT         = None    # 전체 쓰려면 None
# --------------------------------

OUT_DIR.mkdir(parents=True, exist_ok=True)
net = CleanFIDInception(device=DEVICE).eval()

valid = np.load(VAL_NPZ, allow_pickle=True)['arr_0']
if LIMIT is not None:
    valid = valid[:LIMIT]

feats_chunks = []
with torch.inference_mode():
    batch = []
    for img_id, _ in tqdm(valid, desc='Extracting Inception features'):
        p = IMG_ROOT / f'COCO_val2014_{int(img_id):012d}.jpg'
        with Image.open(p) as im:
            im = ImageOps.exif_transpose(im).convert("RGB").copy()
        batch.append(im)

        if len(batch) == BATCH_SIZE:
            feats = net(batch).detach().cpu()           # (B, 2048)
            feats_chunks.append(feats)
            batch.clear()

    # 남은 잔여 배치 처리
    if batch:
        feats = net(batch).detach().cpu()
        feats_chunks.append(feats)

# (N, 2048) float32 -> float64 로 변환해 통계 수치 안정성 확보
feats = torch.cat(feats_chunks, dim=0).to(torch.float64)
mu    = feats.mean(dim=0)                                   # (2048,), float64
# unbiased 공분산(ddof=1). torch.cov는 입력 형태 (D, N) 필요
sigma = torch.cov(feats.T, correction=1)                    # (2048, 2048), float64

torch.save({'mu': mu.cpu(), 'sigma': sigma.cpu(), 'n': int(feats.shape[0])},
           OUT_PATH)

print('done:', mu.shape, sigma.shape, 'N=', int(feats.shape[0]), '->', OUT_PATH)


Extracting Inception features: 100%|██████████| 30000/30000 [11:17<00:00, 44.25it/s]  


done: torch.Size([2048]) torch.Size([2048, 2048]) N= 30000 -> mscoco2014_fid/coco2014_val_30k_fid_stats_clean.pt
